**Release-package note.** The private workstation bootstrap cell was removed. Paths are resolved by `_defs/_init.py`. Before manual use, run `./run prepare-data`; see `README.md` for the tested commands.

In [ ]:
from _defs.iplot import *
from _defs.istats import *

## Monkey 3, gradual adaptation

In [ ]:
animID = 'groot'
dataPath = outputdir+'/cswm_behv_orig_'+animID+'.pkl'
behv_data = load_dictionary_pickle(dataPath)
seqIDs0 = behv_data['seqIDs']
respIDs0 = behv_data['respIDs']
tconstrasts0 = behv_data['tconstrasts']
holdTimes0 = behv_data['holdTimes']

print(seqIDs0.shape,respIDs0.shape,tconstrasts0.shape,holdTimes0.shape)

picks = (seqIDs0[:,1]!=0) # pick long-holding trials

seqIDs = seqIDs0[picks]
respIDs = respIDs0[picks]
tconstrasts = tconstrasts0[picks]
holdTimes = holdTimes0[picks]

print(seqIDs.shape,respIDs.shape,tconstrasts.shape,holdTimes.shape)

picks = respIDs[:,1] != 0 # pick responded trials

seqID = seqIDs[picks]
respID = respIDs[picks]
tconstrast = tconstrasts[picks]

print(seqID.shape, respID.shape, tconstrast.shape)

seqID_m = np.array([seqID[:,1], seqID[:,0]]).T
iscorrs = seqID == respID
iscorrs_m = seqID_m == respID
iscorr = iscorrs.sum(1) == 2
iscorr_m = iscorrs_m.sum(1) == 2
islocerr = (iscorrs.sum(1) != 2) & (iscorrs_m.sum(1) != 2)

win = 40
iscorrd = []
iscorrmd = []
islocerrd = []
for i in range(len(iscorr) - win):
    iscorrd.append(sum(iscorr[i:(i + win)]) / win)
    iscorrmd.append(sum(iscorr_m[i:(i + win)]) / win)
    islocerrd.append(sum(islocerr[i:(i + win)]) / win)

iscorrd = np.array(iscorrd)
iscorrmd = np.array(iscorrmd)
islocerrd = np.array(islocerrd)

# Trial indices for x-axis (centered on window, fixed length match)
x = np.arange(win // 2, len(iscorrd) + win // 2)

# Optimized plotting
redrat = 3.6
fs = 6 * redrat
fig, ax = plt.subplots(1, 1, figsize=[2.5 * redrat, 1.4 * redrat])

# Plot lines
ax.plot(x, iscorrd * 100, label='correct', lw=2, color=colors[0])
ax.plot(x, iscorrmd * 100, label='swap error', lw=2, color=colors[1])
ax.plot(x, islocerrd * 100, label='location error', lw=2, color=colors[2])

# Mark stages with shaded regions and vertical lines
stages = [(1, 199, 'Early'), (202, 599, 'Interim)'), (602, 1400, 'Late')]
colors2 = [colors[2], colors[1], colors[0]]  # Different colors for stages

for i, (start, end, label) in enumerate(stages):
    start_idx = max(0, int((start - 1 - win // 2) / 1))  # Approximate index for start
    end_idx = min(len(x) - 1, int((end - 1 - win // 2) / 1))
    if start_idx < end_idx:
        ax.axvspan(x[start_idx], x[end_idx], alpha=0.3, color=colors2[i])
    if end < len(x):
        boundary_idx = min(len(x) - 1, int((end - win // 2)))
        # ax.axvline(x[boundary_idx], color='black', linestyle='--', alpha=0.5, lw=1)

# Annotations for stage labels (placed at approximate midpoints)
mid_early = int(len(x) * 100 / 1400)  # ~100 for early
mid_interim = int(len(x) * 400 / 1400)  # ~400 for interim
mid_late = int(len(x) * 800 / 1400)  # ~800 for late

ax.text(x[mid_early], 123, 'early', ha='center', va='top', fontsize=fs-1, color='k')
ax.text(x[mid_interim], 123, 'interim', ha='center', va='top', fontsize=fs-1, color='k')
ax.text(x[mid_late]+200, 123, 'post stage', ha='center', va='top', fontsize=fs-1, color='k')

# Legend and styling
ax.legend(frameon=False, loc=[0.16, 0.6], fontsize=fs-7)
ax.set_ylabel('trial %', fontsize=fs)
ax.set_xlabel('trial no.', fontsize=fs)
ax.tick_params(axis='both', which='major', labelsize=fs)
ax.set_ylim([-10, 110])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(True)
ax.spines['left'].set_visible(True)

plt.suptitle('Monkey 3: behavioral dynamics (gradual adaptation)',fontsize=fs)
plt.tight_layout()
plt.show()

# fig.savefig(figdir+'/'+'fig5b.pdf', format='pdf', dpi=300, bbox_inches='tight')

In [ ]:
animID = 'groot'; tempGen = load_dictionary_pickle(outputdir+'/tempGen_cs_'+animID+'.pkl')
hplots_stats(tempGen, title='decode color-shape features (day 1)',vmin=0.2,vmax=0.3,savefigpath=figdir+'/'+'figs7e.pdf',cmap='coolwarm',threshold=0.2)

In [ ]:
animID = 'groot'; tempGen = load_dictionary_pickle(outputdir+'/tempGen_cs_days_'+animID+'.pkl')
tp = tempGen['tp'];tt = tempGen['tt'];ttlabel = tempGen['ttlabel']
times = tempGen['times']

redrat = 3.6; fs=6 * redrat
fig=plt.figure(figsize=[1.8*redrat,2.2*redrat]);
plt.suptitle('decode color-shape features',x=0.55,y=0.96,fontsize=fs)

for i in range(3):
    cross_acc=tempGen['acc'][i]
    
    data = []
    data.append(cross_acc[0].diagonal())
    data.append(cross_acc[3].diagonal())
    data.append(np.concatenate([cross_acc[2].diagonal(offset=int(tp[0]-int(tp[1])))[:int(tp[1])],
                                cross_acc[1].diagonal(offset=int(tp[1]-int(tp[0])))[int(tp[0]):]]))
    
    r1d = np.concatenate([cross_acc[2].diagonal(offset=int(tp[0]-int(tp[1])))[:int(tp[1])],
                          cross_acc[2].diagonal(offset=int(tp[1]-int(tp[0])))[int(tp[0]):]])

    r2d = np.concatenate([cross_acc[1].diagonal(offset=int(tp[0]-int(tp[1])))[:int(tp[1])],
                          cross_acc[1].diagonal(offset=int(tp[1]-int(tp[0])))[int(tp[0]):]])
    # r2d = cross_acc[1].diagonal(offset=int(tp[1]-int(tp[0])))
    ax1 = plt.subplot(3,1,i+1)
    ax1.plot(times,ifilt(data[2],11,0),color='k',label='entry')
    ax1.plot(times,ifilt(data[0]-r1d,11,0)+1/6,color=colors[3],label='rank1')
    ax1.plot(times,ifilt(data[1]-r2d,11,0)+1/6,color=colors[0],label='rank2')

    
    plt.axhline(1/6,ls=':',c='k',label='chance')
    for ii in range(len(tt)-1):
        plt.axvline(tt[ii],c='k',ls=':')
    
    if i==2: plt.xticks(tt[:2],labels=ttlabel[:2],fontsize=fs) 
    else: plt.xticks([])
    if i==0: plt.legend(frameon=False,loc='upper right',fontsize=fs-5)
    plt.ylim([0.13,0.38]);plt.yticks([0.15,0.25,0.35],fontsize=fs-2)
    
    if i ==1: plt.ylabel('decoding accuracy',fontsize=fs)
    # plt.title('day '+str(i+1),fontsize=fs)
    ax2 = ax1.twinx()
    ax2.set_ylabel('day '+str(i+1), color='k',fontsize=fs)
    ax2.set_yticks([])

    plt.gca().spines['top'].set_visible(False);plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['bottom'].set_visible(True);plt.gca().spines['left'].set_visible(True)
plt.tight_layout()

# fig.savefig(figdir+'/'+'figs7f.pdf', format='pdf', dpi=300, bbox_inches='tight')

In [ ]:
recons = load_dictionary_pickle(outputdir+'/cswm_recons2_'+animID+'.pkl')
recons1 = recons['recons1'];recons2 = recons['recons2']
seqIDs = recons['seqIDs'];tconstrasts = recons['tconstrasts']

Params=subjParams('groot')
times=Params['times'];tt = Params['tt'];t2 = Params['t2'];ttlabel = Params['ttlabel'];tp = Params['tp']

# ====================== 参数设置 ======================
suptitle = 'single trial decoding'
llabel = ['T1 location', 'T2 location', 'T1 color-shape', 'T2 color-shape']
fs = 16                    # 整体字体稍增大，更清晰
fs_title = 20
fs_label = 15
fs_tick = 18

nrow = 4
ncol = 3
hgtr = [4.8, 1.0, 4.8, 1.2]      # 略微调整高度比例，让 mean DA 部分更有空间
wdtr = [4.8, 4.5, 0.35]          # 略微增加主图和 colorbar 宽度

kk = 0

fig = plt.figure(figsize=(wdtr[0] + wdtr[1] + wdtr[2] + 0.5,   # 整体增加一点宽度
                          sum(hgtr) + 0.8))                     # 增加顶部空间给 suptitle

gs = gridspec.GridSpec(nrow, ncol, 
                       wspace=0.25, 
                       hspace=0.35,
                       width_ratios=wdtr, 
                       height_ratios=hgtr)

# ====================== 主循环 ======================
for k in range(2):
    if k == 0:
        recons = recons1
        IDs = seqIDs
        vmin = 0.2; vminm=0; vmax = 0.8; vmaxm = 0.8
    else:
        recons = recons2
        IDs = tconstrasts
        vmin = 0.2; vminm=0.1; vmax = 0.4; vmaxm = 0.4
      
    for rid in range(2):
        # 构建 recon 矩阵
        recon = []
        ID = IDs[:, rid]
        for j in range(recons.shape[1]):
            recon.append(recons[int(ID[j]-1), j, :])
        recon = np.array(recon)

        col = kk % 2
        row_img = k * 2
        
        # ==================== 上方 imshow 图 ====================
        ax = plt.subplot(gs[row_img, col])
        ax.set_title(llabel[kk], fontsize=fs_title, pad=12)
        
        im = ax.imshow(recon, 
                       extent=[0, recon.shape[1], recon.shape[0], 0],
                       vmin=vmin, vmax=vmax, 
                       aspect='auto', cmap='viridis')   # 推荐使用 viridis 或 RdYlBu
        
        ax.set_xticks([])
        ax.set_yticks([1, 200, 600, 1000] if rid == 0 else [])
        if rid == 0:
            ax.set_ylabel('Trial No.', fontsize=fs_label)
            ax.set_yticklabels(['1', '200', '600', '1000'], fontsize=fs_tick)
        
        # ==================== Colorbar ====================
        if rid == 1:
            cax = plt.subplot(gs[row_img, 2])
            cbar = fig.colorbar(im, cax=cax, shrink=0.75, aspect=20)
            
            # 去掉 colorbar 边框
            cbar.outline.set_visible(False)
            
            if k == 0:
                cbar.ax.set_yticks([0.2, 0.8])
                cbar.ax.set_yticklabels(['0.2', '0.8'])
            else:
                cbar.ax.set_yticks([0.2, 0.4])
                cbar.ax.set_yticklabels(['0.2', '0.4'])
            
            cbar.ax.tick_params(labelsize=fs_tick)
            cbar.set_label('Decoding Accuracy', 
                          rotation=90, labelpad=15, fontsize=fs_label)
            cbar.ax.yaxis.set_label_position("left")

        # ==================== 下方 mean DA 曲线 ====================
        axl = plt.subplot(gs[row_img + 1, col])
        axl.plot(times, recon.mean(0), c='gray', lw=1.8)
        
        axl.set_ylim([vminm, vmaxm * 1.05])        # 留一点顶部空间
        axl.set_yticks([vminm, vmaxm])
        axl.set_yticklabels(['0', f'{vmaxm}'], fontsize=fs_tick)
        
        axl.set_xlabel('Time (s)' if k == 1 else '', fontsize=fs_label)
        if rid == 0:
            axl.set_ylabel('Mean DA', fontsize=fs_label)
        
        # 美化坐标轴
        axl.spines['top'].set_visible(False)
        axl.spines['right'].set_visible(False)
        axl.spines['left'].set_linewidth(1.2)
        axl.spines['bottom'].set_linewidth(1.2)
        
        axl.axhline(1/6, c='k', ls='--', lw=1.0, alpha=0.7)
        
        # 垂直参考线
        for ii in range(len(tt)-1):
            axl.axvline(tt[ii], c='k', ls='--', lw=1.0, alpha=0.6)
        
        axl.set_xticks(tt[:2])
        axl.set_xticklabels(ttlabel[:2], fontsize=fs_tick)
        
        # 增加网格（可选，更易读）
        # axl.grid(True, axis='y', linestyle=':', alpha=0.3)
        
        kk += 1

# ====================== 全局设置 ======================
# fig.suptitle(suptitle, fontsize=fs_title+2, y=0.98, fontweight='bold')

# 自动调整布局（比 tight_layout 更适合 GridSpec）
plt.subplots_adjust(top=0.92, bottom=0.08, left=0.08, right=0.95)

plt.show()
# fig.savefig(figdir+'/'+'figs7c.pdf', format='pdf', dpi=300, bbox_inches='tight')

In [ ]:
animID = 'groot'
Params=subjParams(animID)
times=Params['times'];tt = Params['tt'];t2 = Params['t2'];ttlabel = Params['ttlabel'];tp = Params['tp']

dataPath = outputdir+'/cswm_behv_'+animID+'.pkl'
behv_data = load_dictionary_pickle(dataPath)
seqIDs0 = behv_data['seqIDs']
respIDs0 = behv_data['respIDs']
tconstrasts0 = behv_data['tconstrasts']
holdTimes0 = behv_data['holdTimes']

dataPath = outputdir+'/cswm_recons_'+animID+'.pkl'
tca = load_dictionary_pickle(dataPath)
recons0 = tca['recons']
recons0 = np.exp(recons0)/np.exp(recons0).sum(axis=0,keepdims=True)
recons0 = ifilt(recons0,7,0)

print(seqIDs0.shape,respIDs0.shape,tconstrasts0.shape,holdTimes0.shape,recons0.shape)

picks = respIDs0[:,1]!=0 # pick responded trials

seqIDs = seqIDs0[picks]
respIDs = respIDs0[picks]
tconstrasts = tconstrasts0[picks]
holdTimes = holdTimes0[picks]
recons = recons0[:,:,picks,:]
print(seqIDs.shape,respIDs.shape,tconstrasts.shape,holdTimes.shape,recons.shape)

In [ ]:
trialsel = 75

picks = respIDs0[:,0]!=0 # pick responded trials

seqIDs = seqIDs0[picks]
respIDs = respIDs0[picks]
tconstrasts = tconstrasts0[picks]
holdTimes = holdTimes0[picks]
recons = recons0[:,:,picks,:]

print(seqIDs.shape,respIDs.shape,tconstrasts.shape,holdTimes.shape,recons.shape)

respIDs_mirror = np.array([respIDs[:,1],respIDs[:,0]]).T
iscorrs_mirror = (seqIDs==respIDs_mirror)
iscorrs = (seqIDs==respIDs)


picks = (iscorrs_mirror.sum(1)!=2) & (iscorrs.sum(1)!=2)
print(np.where(picks==1)[0][:trialsel])
seqIDs = seqIDs[picks]
respIDs = respIDs[picks]
tconstrasts = tconstrasts[picks]
recons = recons[:,:,picks,:]
print(seqIDs.shape,recons.shape)

spt = 'Early Stage (first 100 trials, incorrect report)'
kl=[2,0,0]
# plt.figure(figsize=[7,3]);fs=12
redrat = 3.6; fs=6 * redrat
fig=plt.figure(figsize=[3.8*redrat,1.2*redrat]);
plt.suptitle(spt,x=0.52,y=1.05,fontsize=fs)
colors2 = ['k',colors[3],colors[0]]

reconw = []
for k in range(2):
    reconiNum=[]
    plt.subplot(1,2,k+1)
    plt.title('Target'+str(k+1),fontsize=fs)
    for j in range(3):
        if (j==0) | ((j==1)&(k==0)) | ((j==2)&(k==1)):
        
            seqID = seqIDs[:,k]
            respID = respIDs[:,k]
            recon = recons
    
            recon = recons[:,:,:trialsel,:]

            cmpn = ['entry-seq','rank-seq','rank-seq'];
            ID = seqID[:trialsel]
            reconi=[]
            for l in range(len(np.unique(ID))):
                p = (ID==np.unique(ID)[l])
                reconi.append(recon[l,j,p,:])
            reconi = np.concatenate(reconi,axis=0)
            if j==0: reconw.append(reconi[:,int(tp[k]+5):int(tp[k]+30)].max(1))
            reconiNum.append(len(reconi))
            print(cmpn[j],',',len(reconi),end=', ')
            x=times;y=reconi.mean(0);yerr=reconi.std(0)/np.sqrt(len(reconi))
            plt.plot(x,y,label=cmpn[j],color=colors2[j])
            plt.fill_between(x, y-yerr, y+yerr,color=colors2[j], alpha=0.2)

            if j==0:
                if k==0: sig_y=0.5
                elif k==1: sig_y=0.5
            else:
                if k==0: sig_y=0.3 + 0.05
                elif k==1: sig_y=0.3 + 0.05
            
            # permsplot(reconi,times,1/6,1000,0.01,1,42,0,sig_y)
            tstat,pval= temp_ttest(reconi-1/6,[],ttype='1samp')
            sig_mask = (pval < 0.01) & (tstat > 0) & (times > tt[k])
            plt.scatter(x=times,y=np.ones(len(pval))*sig_y,c=colors2[j],alpha=0.3*sig_mask,marker='o')
            

            cmpn = ['entry-rsp','rank-rsp','rank-rsp'];
            ID = respID[:trialsel]

            reconi=[]
            for l in range(len(np.unique(ID))):
                p = (ID==np.unique(ID)[l])
                reconi.append(recon[l,j,p,:])
            reconi = np.concatenate(reconi,axis=0)
            
                    
                # else: 
                #     reconw.append(reconi[:,int(tp[k]+30):].mean(1))
            reconiNum.append(len(reconi))
            print(cmpn[j],',',len(reconi),end=', ')
            x=times;y=reconi.mean(0);yerr=reconi.std(0)/np.sqrt(len(reconi))
            plt.plot(x,y,label=cmpn[j],color=colors2[j],ls='--')
            plt.fill_between(x, y-yerr, y+yerr,color=colors2[j], alpha=0.2)

            if j==0:
                if k==0: sig_y=0.55
                elif k==1: sig_y=0.55
            else:
                if k==0: sig_y=0.35 + 0.05
                elif k==1: sig_y=0.35 + 0.05

            # permsplot(reconi,times,1/6,1000,0.01,1,42,9,sig_y)
            tstat,pval= temp_ttest(reconi-1/6,[],ttype='1samp')
            sig_mask = (pval < 0.01) & (tstat > 0) & (times > tt[k])
            plt.scatter(x=times,y=np.ones(len(pval))*sig_y,c=colors2[j],alpha=0.3*sig_mask,marker='o',s=6)
            
            if k == 0: 
                plt.ylabel('decoding accuracy',fontsize=fs)
                plt.legend(title='',frameon=False,loc='upper right',fontsize=fs-5);
            elif k==1: plt.legend(title='',frameon=False,loc='upper left',fontsize=fs-5);
            plt.xticks(tt[:2],labels=ttlabel[:2],fontsize=fs)
            
            # for ii in range(len(tt)-1):
            plt.axvline(tt[k],c='k',ls='--',lw=1,alpha=0.3)
            plt.axhline(1/6,c='k',ls='--',lw=1,alpha=0.3)
            # plt.axvline(tt[k]/100,c='k',ls='--',lw=1,alpha=0.3)
            # plt.axvline(tt[0]+13/100,c='k',ls='--',lw=1,alpha=0.3)
            # plt.axvline(tt[1]+13/100,c='k',ls='--',lw=1,alpha=0.3)
            print(';')
            
            plt.ylim([0.1,0.7])
            plt.yticks([0.2,0.4,0.6],fontsize=fs)
            plt.gca().spines['top'].set_visible(False);plt.gca().spines['right'].set_visible(False)
            plt.gca().spines['bottom'].set_visible(True);plt.gca().spines['left'].set_visible(True)
            # plt.ylim([-0.5,2])
            # plt.yticks([0,0.5,1])
reconiNum=np.array(reconiNum)

# fig.savefig(figdir+'/'+'fig5e.pdf', format='pdf', dpi=300, bbox_inches='tight')

In [ ]:
reconw_err = np.concatenate(reconw)
reconw_thres = np.load(outputdir+'/reconw_thres_'+animID+'.npy')
print(reconw_err.shape, reconw_thres.shape,reconw_err.mean(0),reconw_thres.mean(0))

group1 = reconw_err;group2 = reconw_thres;
# 方法1: Welch's t-test（最推荐）
t_stat, p_value = stats.ttest_ind(group1, group2, equal_var=False)
print(f"Welch's t-test: t = {t_stat:.4f}, p-value = {p_value:.4f}")

# 方法2: 标准 t-test（仅供对比，不推荐这里用）
t_stat_pooled, p_pooled = stats.ttest_ind(group1, group2, equal_var=True)

# 方法3: Mann-Whitney U test（如果怀疑不正态）
u_stat, p_mwu = stats.mannwhitneyu(group1, group2, alternative='two-sided')
print(f"Mann-Whitney U: U = {u_stat}, p-value = {p_mwu:.4f}")

# 方法4: Bootstrap 均值差置信区间（推荐报告）
def bootstrap_mean_diff(x, y, n_boot=10000):
    diffs = []
    for _ in range(n_boot):
        sample_x = np.random.choice(x, size=len(x), replace=True)
        sample_y = np.random.choice(y, size=len(y), replace=True)
        diffs.append(np.mean(sample_x) - np.mean(sample_y))
    return np.percentile(diffs, [2.5, 97.5])

ci = bootstrap_mean_diff(group1, group2)
print(f"均值差 95% Bootstrap CI: {ci}")

In [ ]:
picks = respIDs0[:,0]!=0 # pick responded trials

seqIDs = seqIDs0[picks]
respIDs = respIDs0[picks]
tconstrasts = tconstrasts0[picks]
holdTimes = holdTimes0[picks]
recons = recons0[:,:,picks,:]

print(seqIDs.shape,respIDs.shape,tconstrasts.shape,holdTimes.shape,recons.shape)

respIDs_mirror = np.array([respIDs[:,1],respIDs[:,0]]).T
iscorrs_mirror = (seqIDs==respIDs_mirror)

picks = (iscorrs_mirror.sum(1)==2)
print(np.where(picks==1)[0])

seqIDs = seqIDs[picks]
respIDs = respIDs[picks]
tconstrasts = tconstrasts[picks]
recons = recons[:,:,picks,:]
print(seqIDs.shape,recons.shape)


ids=np.where(picks==1)[0]
bins = np.arange(0, max(ids)+50, 50)

plt.figure(figsize=(8, 5))

n, bins, patches = plt.hist(ids, bins=bins,color='gray', alpha=0.8)

for i in range(len(bins)-1):
    bin_center = (bins[i] + bins[i+1]) / 2
    if 200 <= bin_center <= 600:
        patches[i].set_facecolor('k')      # 可换成 'orange', '#FF7F50', 'crimson' 等
        patches[i].set_alpha(0.5)

plt.xlabel('Trial No.', fontsize=12)
plt.ylabel('Number of Trials', fontsize=12)
plt.title('Distribution of Swap Error Trials\n(Total = {})'.format(len(ids)), 
          fontsize=14, pad=15)
plt.gca().spines['top'].set_visible(False);plt.gca().spines['right'].set_visible(False)
plt.gca().spines['bottom'].set_visible(True);plt.gca().spines['left'].set_visible(True)
plt.tight_layout()
plt.show()

In [ ]:
ids = np.where(picks==1)[0]
# sel = (ids<600) & (ids>200)
sel = (ids<600) & (ids>200)

spt = 'Emerging Order (trial No.200-600)'
kl=[2,0,0]
# plt.figure(figsize=[7,3]);fs=12
redrat = 3.6; fs=6 * redrat
fig=plt.figure(figsize=[3.8*redrat,1.2*redrat]);
plt.suptitle(spt,x=0.52,y=1.05,fontsize=fs)
colors2 = ['k',colors[3],colors[0]]

for k in range(2):
    if k==0:
        cmpn = ['entry','T1@R1','T1@R2'];
    elif k==1:
        cmpn = ['entry','T2@R1','T2@R2'];
    reconiNum=[]
    plt.subplot(1,2,k+1)
    plt.title('Target'+str(k+1),fontsize=fs)
    for j in range(3):
        
        
        seqID = seqIDs[sel,k]
        respID = respIDs[sel,k]
        recon = recons[:,:,sel,:]
        
        reconi=[]
        for l in range(len(np.unique(seqID))):
            p = (seqID==np.unique(seqID)[l])
            reconi.append(recon[l,j,p,:])
            
        reconi = np.concatenate(reconi,axis=0)
        reconiNum.append(len(reconi))
        print(cmpn[j],',',len(reconi),end=', ')
        x=times;y=reconi.mean(0);yerr=reconi.std(0)/np.sqrt(len(reconi))
        plt.plot(x,y,label=cmpn[j],color=colors2[j])
        plt.fill_between(x, y-yerr, y+yerr,color=colors2[j], alpha=0.2)

        if j==0:
            if k==0: sig_y=0.48
            elif k==1: sig_y=0.48
        else:
            if k==0: sig_y=0.35 + j*0.05
            elif k==1: sig_y=0.35 + j*0.05
        
        tstat,pval= temp_ttest(reconi-1/6,[],ttype='1samp')
        sig_mask = (pval < 0.01) & (tstat > 0) & (times > tt[k])
        plt.scatter(x=times,y=np.ones(len(pval))*sig_y,c=colors2[j],alpha=0.3*sig_mask,marker='o')
        
        if k == 0: plt.ylabel('decoding accuracy',fontsize=fs)
        if j == 2: plt.xticks(tt[:2],labels=ttlabel[:2],fontsize=fs)
        else: plt.xticks([])
        if k == 0: plt.legend(title='',frameon=False,loc='upper right',fontsize=fs-4);
        elif k==1: plt.legend(title='',frameon=False,loc='upper left',fontsize=fs-4);
        # for ii in range(len(tt)-1):
        plt.axvline(tt[k],c='k',ls='--',lw=1,alpha=0.3)
        plt.axhline(1/6,c='k',ls='--',lw=1,alpha=0.3)
        print(';')
        
        plt.ylim([0.05,0.7])
        plt.yticks([0.2,0.4,0.6],fontsize=fs)
        plt.gca().spines['top'].set_visible(False);plt.gca().spines['right'].set_visible(False)
        plt.gca().spines['bottom'].set_visible(True);plt.gca().spines['left'].set_visible(True)
        # plt.yticks([0,0.5,1])
reconiNum=np.array(reconiNum)
# fig.savefig(figdir+'/'+'fig5f.pdf', format='pdf', dpi=300, bbox_inches='tight')

In [ ]:
Params=subjParams('groot')
times=Params['times'];tt = Params['tt'];t2 = Params['t2'];ttlabel = Params['ttlabel'];tp = Params['tp']

reconsDA = load_dictionary_pickle(outputdir+'/cswm_reconsDA_'+animID+'.pkl')
seqIDs_L = reconsDA['seqIDs_L'];respIDs_L = reconsDA['respIDs_L'];recons_L = reconsDA['recons_L'];


spt = 'Post Stage'
kl=[2,0,0]

redrat = 3.6; fs=6 * redrat
fig=plt.figure(figsize=[3.8*redrat,1.2*redrat]);
plt.suptitle(spt,x=0.5,y=1.04,fontsize=fs)
colors2 = ['k',colors[3],colors[0]]
reconw=[]
for k in range(2):
    if k==0:
        cmpn = ['entry','T1@R1','T1@R2'];
    elif k==1:
        cmpn = ['entry','T2@R1','T2@R2'];
    reconiNum=[]
    plt.subplot(1,2,k+1)
    plt.title('Target'+str(k+1),fontsize=fs)
    for j in range(3):
        
        p = (respIDs_L[k]!=0)&(seqIDs_L[k]!=0)
        seqIDs = seqIDs_L[k][p]
        respIDs = respIDs_L[k][p]
        recons = recons_L[k][:,:,p,:]

        p=seqIDs==respIDs
        seqID = seqIDs[p]
        respID = respIDs[p]
        recon = recons[:,:,p,:]
        recon = ifilt(recon,7,0)
        
        reconi=[]
        for l in range(len(np.unique(seqID))):
            p = (seqID==np.unique(seqID)[l])
            reconi.append(recon[l,j,p,:])
            
        reconi = np.concatenate(reconi,axis=0)
        if j==0: reconw.append(reconi[:,int(tp[k]+5):int(tp[k]+30)].max(1))
            
        reconiNum.append(len(reconi))
        print(cmpn[j],',',len(reconi),end=', ')
        x=Params['times'];y=reconi.mean(0);yerr=reconi.std(0)/np.sqrt(len(reconi))
        plt.plot(x,y,label=cmpn[j],color=colors2[j])
        plt.fill_between(x, y-yerr, y+yerr,color=colors2[j], alpha=0.2)

        if j==0:
            if k==0: sig_y=0.58
            elif k==1: sig_y=0.58
        else:
            if k==0: sig_y=0.4 + j*0.05
            elif k==1: sig_y=0.4 + j*0.05
        
        tstat,pval= temp_ttest(reconi-1/6,[],ttype='1samp')
        sig_mask = (pval < 0.01) & (tstat > 0) & (times > tt[k])
        plt.scatter(x=times,y=np.ones(len(pval))*sig_y,c=colors2[j],alpha=0.3*sig_mask,marker='o')
        
        if k == 0: plt.ylabel('decoding accuracy',fontsize=fs)
        if j == 2: plt.xticks(Params['tt'][:2],labels=Params['ttlabel'][:2],fontsize=fs)
        else: plt.xticks([])
        if k == 0: plt.legend(title='',frameon=False,loc='upper right',fontsize=fs-4);
        elif k==1: plt.legend(title='',frameon=False,loc='upper left',fontsize=fs-4);
        # for ii in range(len(tt)-1):
        plt.axvline(Params['tt'][k],c='k',ls='--',lw=1,alpha=0.3)
        # plt.axvline(tt[k]+8/100,c='k',ls='--',lw=1,alpha=0.3)
        # plt.axvline(tt[k]+18/100,c='k',ls='--',lw=1,alpha=0.3)
        plt.axhline(1/6,c='k',ls='--',lw=1,alpha=0.3)
        print(';')
        
        plt.ylim([0.05,0.7])
        plt.yticks([0.2,0.4,0.6],fontsize=fs)
        plt.gca().spines['top'].set_visible(False);plt.gca().spines['right'].set_visible(False)
        plt.gca().spines['bottom'].set_visible(True);plt.gca().spines['left'].set_visible(True)
        # plt.yticks([0,0.5,1])
reconiNum=np.array(reconiNum)

# fig.savefig(figdir+'/'+'fig5g.pdf', format='pdf', dpi=300, bbox_inches='tight')

In [ ]:
reconw_err = np.concatenate(reconw)
reconw_thres = np.load(outputdir+'/reconw_thres_'+animID+'.npy')
print(reconw_err.shape, reconw_thres.shape,reconw_err.mean(0),reconw_thres.mean(0))

group1 = reconw_err;group2 = reconw_thres;
# 方法1: Welch's t-test（最推荐）
t_stat, p_value = stats.ttest_ind(group1, group2, equal_var=False)
print(f"Welch's t-test: t = {t_stat:.4f}, p-value = {p_value:.4f}")

# 方法2: 标准 t-test（仅供对比，不推荐这里用）
t_stat_pooled, p_pooled = stats.ttest_ind(group1, group2, equal_var=True)

# 方法3: Mann-Whitney U test（如果怀疑不正态）
u_stat, p_mwu = stats.mannwhitneyu(group1, group2, alternative='two-sided')
print(f"Mann-Whitney U: U = {u_stat}, p-value = {p_mwu:.4f}")

# 方法4: Bootstrap 均值差置信区间（推荐报告）
def bootstrap_mean_diff(x, y, n_boot=10000):
    diffs = []
    for _ in range(n_boot):
        sample_x = np.random.choice(x, size=len(x), replace=True)
        sample_y = np.random.choice(y, size=len(y), replace=True)
        diffs.append(np.mean(sample_x) - np.mean(sample_y))
    return np.percentile(diffs, [2.5, 97.5])

ci = bootstrap_mean_diff(group1, group2)
print(f"均值差 95% Bootstrap CI: {ci}")

## Monkey 2, few-shot learning

In [ ]:
animID = '2019'

dataPath = outputdir+'/cswm_behv_'+animID+'.pkl'
behv_data = load_dictionary_pickle(dataPath)
seqIDs0 = behv_data['seqIDs']
respIDs0 = behv_data['respIDs']
tconstrasts0 = behv_data['tconstrasts']
holdTimes0 = behv_data['holdTimes']

print(seqIDs0.shape,respIDs0.shape,tconstrasts0.shape,holdTimes0.shape)

picks = (seqIDs0[:,1]!=0) # pick long-holding trials

seqIDs = seqIDs0[picks]
respIDs = respIDs0[picks]
tconstrasts = tconstrasts0[picks]
holdTimes = holdTimes0[picks]

print(seqIDs.shape,respIDs.shape,tconstrasts.shape,holdTimes.shape)

picks = respIDs[:,1] != 0 # pick responded trials

seqID = seqIDs[picks]
respID = respIDs[picks]
tconstrast = tconstrasts[picks]

print(seqID.shape, respID.shape, tconstrast.shape)

seqID_m = np.array([seqID[:,1], seqID[:,0]]).T
iscorrs = seqID == respID
iscorrs_m = seqID_m == respID
iscorr = iscorrs.sum(1) == 2
iscorr_m = iscorrs_m.sum(1) == 2
islocerr = (iscorrs.sum(1) != 2) & (iscorrs_m.sum(1) != 2)

win = 30
iscorrd = []
iscorrmd = []
islocerrd = []
for i in range(len(iscorr) - win):
    iscorrd.append(sum(iscorr[i:(i + win)]) / win)
    iscorrmd.append(sum(iscorr_m[i:(i + win)]) / win)
    islocerrd.append(sum(islocerr[i:(i + win)]) / win)

iscorrd = np.array(iscorrd)
iscorrmd = np.array(iscorrmd)
islocerrd = np.array(islocerrd)

# Trial indices for x-axis (centered on window, fixed length match)
x = np.arange(win // 2, len(iscorrd) + win // 2)

# Optimized plotting
redrat = 3.6
fs = 6 * redrat
fig, ax = plt.subplots(1, 1, figsize=[2.5 * redrat, 1.4 * redrat])

# Plot lines
ax.plot(x, iscorrd * 100, label='Correct', lw=2, color=colors[0])
ax.plot(x, iscorrmd * 100, label='Swap error', lw=2, color=colors[1])
ax.plot(x, islocerrd * 100, label='Location error', lw=2, color=colors[2])

# Mark stages with shaded regions and vertical lines
stages = [(1, 149, 'Early'), (153, 1400, 'Late)')]
colors2 = [colors[2], colors[0], colors[0]]  # Different colors for stages

for i, (start, end, label) in enumerate(stages):
    start_idx = max(0, int((start - 1 - win // 2) / 1))  # Approximate index for start
    end_idx = min(len(x) - 1, int((end - 1 - win // 2) / 1))
    if start_idx < end_idx:
        ax.axvspan(x[start_idx], x[end_idx], alpha=0.3, color=colors2[i])
    if end < len(x):
        boundary_idx = min(len(x) - 1, int((end - win // 2)))
        # ax.axvline(x[boundary_idx], color='black', linestyle='--', alpha=0.5, lw=1)

# Annotations for stage labels (placed at approximate midpoints)
mid_early = int(len(x) * 100 / 1400)  # ~100 for early
mid_interim = int(len(x) * 400 / 1400)  # ~400 for interim
mid_late = int(len(x) * 800 / 1400)  # ~800 for late

ax.text(x[mid_early], 123, 'Early', ha='center', va='top', fontsize=fs-1, color='k')
ax.text(x[mid_interim], 123, 'Post stage', ha='center', va='top', fontsize=fs-1, color='k')
# ax.text(x[mid_late]+200, 123, 'Post stage', ha='center', va='top', fontsize=fs-1, color='k')

# Legend and styling
ax.legend(frameon=False, loc=[0.17, 0.4], fontsize=fs-7)
ax.set_ylabel('Trial %', fontsize=fs)
ax.set_xlabel('Trial no.', fontsize=fs)
ax.tick_params(axis='both', which='major', labelsize=fs)
ax.set_ylim([-10, 110])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(True)
ax.spines['left'].set_visible(True)

plt.suptitle('Monkey 2: behavioral dynamics (few-shot learning)',fontsize=fs)
plt.tight_layout()
plt.show()

# fig.savefig(figdir+'/'+'figs7a.pdf', format='pdf', dpi=300, bbox_inches='tight')

In [ ]:
animID = '2019'
Params=subjParams(animID)
times=Params['times'];tt = Params['tt'];t2 = Params['t2'];ttlabel = Params['ttlabel'];tp = Params['tp']

dataPath =  outputdir+'/cswm_behv_'+animID+'.pkl'
behv_data = load_dictionary_pickle(dataPath)
seqIDs0 = behv_data['seqIDs']
respIDs0 = behv_data['respIDs']
tconstrasts0 = behv_data['tconstrasts']
holdTimes0 = behv_data['holdTimes']


dataPath = outputdir+'/cswm_recons_'+animID+'.pkl'
tca = load_dictionary_pickle(dataPath)
recons0 = tca['recons']
recons0 = np.exp(recons0)/np.exp(recons0).sum(axis=0,keepdims=True)
# recons0 = ifilt(recons0,21,0)

print(seqIDs0.shape,respIDs0.shape,tconstrasts0.shape,holdTimes0.shape,recons0.shape)

picks = respIDs0[:,1]!=0 # pick responded trials

seqIDs = seqIDs0[picks]
respIDs = respIDs0[picks]
tconstrasts = tconstrasts0[picks]
holdTimes = holdTimes0[picks]
recons = recons0[:,:,picks,:]
print(seqIDs.shape,respIDs.shape,tconstrasts.shape,holdTimes.shape,recons.shape)


# trialsel = 30
trialsel = 15
# trialsel = -1 # all error trials


picks = respIDs0[:,0]!=0 # pick responded trials

seqIDs = seqIDs0[picks]
respIDs = respIDs0[picks]
tconstrasts = tconstrasts0[picks]
holdTimes = holdTimes0[picks]
recons = recons0[:,:,picks,:]

print(seqIDs.shape,respIDs.shape,tconstrasts.shape,holdTimes.shape,recons.shape)

respIDs_mirror = np.array([respIDs[:,1],respIDs[:,0]]).T
iscorrs_mirror = (seqIDs==respIDs_mirror)
iscorrs = (seqIDs==respIDs)


picks = (iscorrs.sum(1)!=2)
print(np.where(picks==1)[0][:trialsel])
seqIDs = seqIDs[picks]
respIDs = respIDs[picks]
tconstrasts = tconstrasts[picks]
recons = recons[:,:,picks,:]
print(seqIDs.shape,recons.shape)

spt = 'early error trials'
kl=[2,0,0]
# plt.figure(figsize=[7,3]);fs=12
redrat = 3.6; fs=6 * redrat
fig=plt.figure(figsize=[2.5*redrat,1.4*redrat]);
plt.suptitle(spt,x=0.52,y=1.05,fontsize=fs)
colors2 = ['k',colors[3],colors[0]]

reconw = []
for k in range(2):
    reconiNum=[]
    plt.subplot(1,2,k+1)
    plt.title('Target'+str(k+1),fontsize=fs)
    for j in range(3):
        if (j==0) | ((j==1)&(k==0)) | ((j==2)&(k==1)):
        
            seqID = seqIDs[:,k]
            respID = respIDs[:,k]
            recon = recons
    
            recon = recons[:,:,:trialsel,:]

            cmpn = ['entry-seq','rank-seq','rank-seq'];
            ID = seqID[:trialsel]
            reconi=[]
            for l in range(len(np.unique(ID))):
                p = (ID==np.unique(ID)[l])
                reconi.append(recon[l,j,p,:])
            reconi = np.concatenate(reconi,axis=0)
        
            reconiNum.append(len(reconi))
            print(cmpn[j],',',len(reconi),end=', ')
            x=times;y=reconi.mean(0);yerr=reconi.std(0)/np.sqrt(len(reconi))
            plt.plot(x,y,label=cmpn[j],color=colors2[j])
            plt.fill_between(x, y-yerr, y+yerr,color=colors2[j], alpha=0.2)

            if j==0:
                if k==0: sig_y=0.65
                elif k==1: sig_y=0.65
            else:
                if k==0: sig_y=0.68
                elif k==1: sig_y=0.7
            
            # permsplot(reconi,times,1/6,1000,0.01,1,42,0,sig_y)
            tstat,pval= temp_ttest(reconi-1/6,[],ttype='1samp')
            sig_mask = (pval < 0.01) & (tstat > 0) & (times > tt[k])
            plt.scatter(x=times,y=np.ones(len(pval))*sig_y,c=colors2[j],alpha=0.3*sig_mask,marker='o')
            

            cmpn = ['entry-rsp','rank-rsp','rank-rsp'];
            ID = respID[:trialsel]

            reconi=[]
            for l in range(len(np.unique(ID))):
                p = (ID==np.unique(ID)[l])
                reconi.append(recon[l,j,p,:])
            reconi = np.concatenate(reconi,axis=0)

            reconiNum.append(len(reconi))
            print(cmpn[j],',',len(reconi),end=', ')
            x=times;y=reconi.mean(0);yerr=reconi.std(0)/np.sqrt(len(reconi))
            plt.plot(x,y,label=cmpn[j],color=colors2[j],ls='--')
            plt.fill_between(x, y-yerr, y+yerr,color=colors2[j], alpha=0.2)

            if j==0:
                if k==0: sig_y=0.6
                elif k==1: sig_y=0.6
            else:
                if k==0: sig_y=0.57
                elif k==1: sig_y=0.65
            # permsplot(reconi,times,1/6,1000,0.01,1,42,9,sig_y)
            tstat,pval= temp_ttest(reconi-1/6,[],ttype='1samp')
            sig_mask = (pval < 0.01) & (tstat > 0) & (times > tt[k])
            plt.scatter(x=times,y=np.ones(len(pval))*sig_y,c=colors2[j],alpha=0.3*sig_mask,marker='o',s=6)
            
            if k == 0: 
                plt.ylabel('decoding accuracy',fontsize=fs)
                plt.legend(title='',frameon=False,loc='upper right',fontsize=fs-9);
            elif k==1: plt.legend(title='',frameon=False,loc='upper left',fontsize=fs-9);
            plt.xticks(tt[:2],labels=ttlabel[:2],fontsize=fs)
            
            # for ii in range(len(tt)-1):
            plt.axvline(tt[k],c='k',ls='--',lw=1,alpha=0.3)
            plt.axhline(1/6,c='k',ls='--',lw=1,alpha=0.3)
            print(';')
            
            plt.ylim([0,1])
            plt.yticks([0.2,0.5,0.8],fontsize=fs)
            plt.gca().spines['top'].set_visible(False);plt.gca().spines['right'].set_visible(False)
            plt.gca().spines['bottom'].set_visible(True);plt.gca().spines['left'].set_visible(True)
            # plt.ylim([-0.5,2])
            # plt.yticks([0,0.5,1])
reconiNum=np.array(reconiNum)

# fig.savefig(figdir+'/'+'figs7b.pdf', format='pdf', dpi=300, bbox_inches='tight')